In [0]:
from pyspark.sql import functions as F

In [0]:
# silver sales
SOURCE_CATALOG_NAME_SS = 'beverage_sales'
SOURCE_SCHEMA_NAME_SS = 'silver'
SOURCE_TABLE_NAME_SS = 'sales'

# gold dimensions
SOURCE_CATALOG_NAME_GOLD = 'beverage_sales'
SOURCE_SCHEMA_NAME_GOLD = 'gold'

# fact_sales
TARGET_CATALOG_NAME = 'beverage_sales'
TARGET_SCHEMA_NAME = 'gold'
TARGET_TABLE_NAME = 'fact_sales'

UNKNOWN_KEY = -1

In [0]:
df_silver_sales = spark.table(f'{SOURCE_CATALOG_NAME_SS}.{SOURCE_SCHEMA_NAME_SS}.{SOURCE_TABLE_NAME_SS}')
df_dim_brand = spark.table(f'{SOURCE_CATALOG_NAME_GOLD}.{SOURCE_SCHEMA_NAME_GOLD}.dim_brand')
df_dim_region = spark.table(f'{SOURCE_CATALOG_NAME_GOLD}.{SOURCE_SCHEMA_NAME_GOLD}.dim_region')
df_dim_channel = spark.table(f'{SOURCE_CATALOG_NAME_GOLD}.{SOURCE_SCHEMA_NAME_GOLD}.dim_channel')
df_dim_package = spark.table(f'{SOURCE_CATALOG_NAME_GOLD}.{SOURCE_SCHEMA_NAME_GOLD}.dim_package')

In [0]:
df_fact_sales = (
    df_silver_sales.alias('ss')
    .join(
        F.broadcast(df_dim_brand.select('brand_key', 'brand_code')),
        on='brand_code',
        how='left'
    )
    .join(
        F.broadcast(df_dim_region.select('region_key', 'region')),
        on='region',
        how='left'
    )
    .join(
        F.broadcast(df_dim_channel.select('channel_key', 'trade_channel')),
        on='trade_channel',
        how='left'
    )
    .join(
        F.broadcast(df_dim_package.select('package_key', 'package_name')),
        on='package_name',
        how='left'
    )
    .select(
        F.date_format('sales_date', 'yyyyMMdd').cast('int').alias('date_key'),
        F.coalesce(F.col('brand_key'), F.lit(UNKNOWN_KEY)).alias('brand_key'),
        F.coalesce(F.col('region_key'), F.lit(UNKNOWN_KEY)).alias('region_key'),
        F.coalesce(F.col('channel_key'), F.lit(UNKNOWN_KEY)).alias('channel_key'),
        F.coalesce(F.col('package_key'), F.lit(UNKNOWN_KEY)).alias('package_key'),

        F.col('period'),
        F.col('dollar_volume'),
        F.col('is_negative_volume')
    )
)

In [0]:
df_fact_sales\
    .write\
    .mode('overwrite')\
    .saveAsTable(f'{TARGET_CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}')